In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

from geo_data import data_handler, helpers
from geo_data.svg_handler import MapSVG


COLORS = {
    "background": "#f6f6f6",
    "border": "#646464",
    "land": "#fefee9",
    "river": "#0978ab",
    "lake": "#c6ecff"
}

# TODO:
# das hier kann man vielleicht benutzen, um es nicht ganz so detailreich zu machen:
# germany["geometry"] = germany["geometry"].simplify(0.01, preserve_topology=True)

# Load data

In [ ]:
countries = data_handler.load(kind="country", source="ne", resolution=10)
all_states = data_handler.load(kind="state", source="ne", resolution=10)
all_rivers = data_handler.load(kind="river", source="ne", resolution=10, identifier="name_de")

germany = countries[countries["name"] == "Germany"]
states = all_states[all_states["admin"] == "Germany"]
rivers = gpd.overlay(all_rivers, germany, how="intersection")
rivers.columns = rivers.columns.str.removesuffix("_1")  # gdp.overlay adds weird suffix

# Plot data and create an SVG-file with matplotlib

In [ ]:
fig, ax = plt.subplots()
fig.set_facecolor(COLORS["background"])
ax.set_axis_off()

# boundaries
germany.plot(ax=ax, color=COLORS["land"], zorder=1)
germany.boundary.plot(ax=ax, color=COLORS["border"], linewidth=1.2, zorder=2)
states.boundary.plot(ax=ax, color=COLORS["border"], linewidth=0.4, zorder=3)
rivers.plot(ax=ax, color=COLORS["river"], linewidth=0.5, zorder=4)

result_path = helpers.get_top_directory() / "results" / "Germany_location_map_plt.svg"
plt.savefig(result_path, format="svg", bbox_inches="tight")

# Create SVG-file with the svg_handler

In [ ]:
geom_germany = germany.iloc[0].geometry

bounds = geom_germany.bounds
x_min, y_min, x_max, y_max = bounds 
x_range, y_range = x_max - x_min, y_max - y_min

height = 500
width = int(y_range / x_range * height)

canvas = MapSVG(size=(width, height), bounds=bounds)

# german outer bounds
outer_bound_kwargs = canvas.get_kwargs("land")
outer_bound_kwargs["stroke_width"] = 3
canvas.add_gdf(germany, "Countries", **outer_bound_kwargs)

# state bounds
canvas.add_gdf(states, "States", **canvas.get_kwargs("land"))

# rivers
canvas.add_gdf(rivers, "Rivers", **canvas.get_kwargs("river"))

result_path = helpers.get_top_directory() / "results" / "Germany_location_map.svg"
canvas.save(result_path)
canvas